# Experiment 013 Ablations: Corpus-Only (013c) and Blind-Only (013d)

Tests which CSQE component drives the gain:
- **Exp 013c (corpus-only):** 4 corpus + 0 blind samples — pure corpus grounding
- **Exp 013d (blind-only):** 0 corpus + 4 blind samples — pure blind QE (like Query2Doc ×4)

Both use the same model, first-pass results, and α=4 query repetition as exp_013.
Only `num_corpus_samples` and `num_blind_samples` change.

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets accelerate bitsandbytes huggingface_hub
!pip install -q bm25s PyStemmer nltk
!pip install -q --upgrade pillow

print('=' * 60)
print('Installation complete')
print('=' * 60)
print('IMPORTANT: Restart runtime now!')
print('  1. Runtime -> Restart runtime')
print('  2. Then run cells starting from Step 2 below')
print('=' * 60)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

import os, sys
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# CRITICAL: Prevent TensorFlow from silently pre-allocating all GPU VRAM.
# Pyserini/BM25S imports can trigger TF in the background. Without this,
# TF grabs ~30GB on A100, leaving no room for the Aya model.
# This was NOT needed in the original Aya notebook because BM25 was loaded
# in a separate evaluation notebook -- but CSQE needs BM25 during generation.
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

# Setup symlinks for BM25S index (adjust drive_base if your path differs)
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
!mkdir -p data/miracl_ar
!ln -sf "{drive_base}/bm25s_index" data/miracl_ar/bm25s_index
!ln -sf "{drive_base}/corpus_ids.pkl" data/miracl_ar/corpus_ids.pkl

print('Environment configured')

In [ ]:
from huggingface_hub import login

# Login to HuggingFace (prompted for token)
login()

print('Logged in to HuggingFace')
print('Make sure you have accepted the Aya Expanse license at:')
print('https://huggingface.co/CohereForAI/aya-expanse-8b')


In [ ]:
import os, sys, json, pickle, time, re
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.utils.data_loader import MIRACLDataLoader

# NOTE: BM25SRetriever and evaluation are in a SEPARATE notebook
# (evaluate_enhanced_queries.ipynb), not here.

print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


In [ ]:
# Shared base config — same as exp_013 except exp_id/output/checkpoint
CONFIG_BASE = {
    'bm25_index_path': 'data/miracl_ar/bm25s_index',
    'bm25_corpus_ids_path': 'data/miracl_ar/corpus_ids.pkl',
    'top_k_docs': 5,
    'doc_truncation_tokens': 128,
    'model_name': 'CohereForAI/aya-expanse-8b',
    'temperature': 1.0,
    'max_new_tokens': 128,
    'top_p': 0.9,
    'query_repetition': 4,
    'corpus_batch_size': 8,
    'blind_batch_size': 32,
    'max_corpus_prompt_length': 2048,
    'max_blind_prompt_length': 512,
    'macro_batch_size': 200,
    'corpus_dict_drive_path': '/content/drive/MyDrive/graduation project/colab_data/corpus_dict.pkl',
}
print('Base config loaded')

In [ ]:
data_loader = MIRACLDataLoader(language='ar', split='dev')
topics, qrels = data_loader.load_all()

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f'Queries: {len(query_ids)}')
print(f'Qrels: {len(qrels)}')
print(f'Sample query: {query_texts[0]}')


In [ ]:
# Load MIRACL corpus for document text lookup
# BM25S index does NOT store original text (only tokenized form)
# We need the original text to pass retrieved docs to the LLM
#
# Uses direct JSONL.gz download (same approach as bm25s_baseline.ipynb)
# This avoids the UnicodeDecodeError from HuggingFace datasets loader

import gzip, requests

corpus_dict_path = CONFIG_BASE['corpus_dict_drive_path']

if os.path.exists(corpus_dict_path):
    print(f'Loading corpus_dict from Drive cache: {corpus_dict_path}')
    with open(corpus_dict_path, 'rb') as f:
        corpus_dict = pickle.load(f)
    print(f'Loaded {len(corpus_dict):,} documents from cache')
else:
    print('corpus_dict not cached on Drive. Downloading MIRACL corpus chunks...')
    base_url = "https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ar/docs-{}.jsonl.gz"
    num_chunks = 5

    corpus_dict = {}
    for chunk_idx in range(num_chunks):
        file_url = base_url.format(chunk_idx)
        temp_file = f'/tmp/miracl_ar_docs_{chunk_idx}.jsonl.gz'

        print(f'  Chunk {chunk_idx+1}/{num_chunks}: downloading...')
        response = requests.get(file_url, stream=True)
        with open(temp_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)

        print(f'  Chunk {chunk_idx+1}/{num_chunks}: parsing...')
        with gzip.open(temp_file, 'rt', encoding='utf-8') as f:
            for line in f:
                doc = json.loads(line)
                title = doc.get('title', '').strip()
                text = doc.get('text', '').strip()
                corpus_dict[doc['docid']] = f'{title} {text}'.strip() if title else text

        os.remove(temp_file)
        print(f'  Chunk {chunk_idx+1}/{num_chunks}: done ({len(corpus_dict):,} docs so far)')

    print('Saving corpus_dict to Drive for future reuse...')
    os.makedirs(os.path.dirname(corpus_dict_path), exist_ok=True)
    with open(corpus_dict_path, 'wb') as f:
        pickle.dump(corpus_dict, f)
    print(f'Saved to {corpus_dict_path}')

print(f'corpus_dict ready: {len(corpus_dict):,} documents')
sample_docid = list(corpus_dict.keys())[0]
print(f'Sample [{sample_docid}]: {corpus_dict[sample_docid][:100]}')

In [ ]:
# Load pre-computed BM25 first-pass results from Drive
# Original was k=10. We now use top_k_docs=5 for faster prompts.
# Just slice the top-5 from the cached k=10 results (no BM25S needed).

firstpass_path = '/content/drive/MyDrive/exp_013_firstpass.pkl'

if os.path.exists(firstpass_path):
    print(f'Loading first-pass results from {firstpass_path}')
    with open(firstpass_path, 'rb') as f:
        firstpass_results_raw = pickle.load(f)
    # Trim to top_k_docs (cached may have more)
    k = CONFIG_BASE['top_k_docs']
    firstpass_results = {
        qid: docs[:k] for qid, docs in firstpass_results_raw.items()
    }
    del firstpass_results_raw
    print(f'Loaded {len(firstpass_results)} queries, trimmed to top-{k} docs each')
else:
    raise FileNotFoundError(
        f'First-pass results not found at {firstpass_path}. '
        'Run the BM25 first-pass cell first (see notebook history).'
    )

# Verify
sample_qid = query_ids[0]
sample_docs = firstpass_results[sample_qid]
print(f'\nSample: qid={sample_qid}, {len(sample_docs)} docs')
print(f'  Top doc: {sample_docs[0]["docid"]} (score={sample_docs[0]["score"]:.4f})')
print(f'  Text preview: {sample_docs[0]["text"][:80]}')


In [ ]:
def parse_docid(docid):
    """Parse X#Y format to article_id and passage_position."""
    parts = docid.split('#')
    return {
        'article_id': int(parts[0]),
        'passage_pos': int(parts[1]) if len(parts) > 1 else 0
    }


def truncate_to_tokens(text, max_tokens=128, tokenizer=None):
    """
    Truncate text to approximately max_tokens.
    Uses tokenizer if available, else char-based fallback.
    """
    if not text:
        return ''
    if tokenizer is not None:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
            return tokenizer.decode(tokens, skip_special_tokens=True)
        return text
    else:
        # Rough fallback: 128 tokens â‰ˆ 512 chars for Arabic
        return text[:512]


print('Helper functions defined')


In [ ]:
# SOURCE: Paper arxiv 2402.18031 Tables 1 & 2, Section 3.1
# English instructions (same pattern as all our existing Aya notebooks).
# Arabic output enforced via 'Respond in Arabic only.'

# â”€â”€ System prompts â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BLIND_SYSTEM = (
    "You are asked to write a passage that answers the given query. "
    "Do not ask the user for further clarification. "
    "Respond in Arabic only."
)

CSQE_SYSTEM = (
    "You are an information retrieval assistant. "
    "You will examine retrieved documents and extract key sentences relevant to the query. "
    "The documents are in Arabic. Respond in Arabic only."
)

# â”€â”€ One-shot example (paper Table 2, English query/docs) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Aya 8B is multilingual â€” the English example generalises to Arabic queries.
CSQE_ONE_SHOT = (
    'Query: "how are some sharks warm blooded"\n'
    'Retrieved documents:\n'
    '1. Most sharks are cold-blooded. Some, like the Mako and the Great white shark, '
    'are partially warm-blooded (they are endotherms)...\n'
    '2. Are sharks cold-blooded or warm-blooded? Sharks have a reputation as cold-blooded...\n'
    '3. Great white sharks are some of the only warm blooded sharks...\n'
    'You will begin by examining the initially retrieved documents and identifying the ones '
    'that are relevant, even partially, to the query. Once the relevant documents are '
    'identified, you will extract the key sentences from each document that contribute '
    'to their relevance.\n'
    'Based on the query "how are some sharks warm blooded", I have examined the initially '
    'retrieved documents. Here are the relevant documents and the key sentences extracted '
    'from each:\n'
    'Document 1:\n'
    '"Most sharks are cold-blooded. Some, like the Mako and the Great white shark, '
    'are partially warm-blooded (they are endotherms)."\n'
    'Document 3:\n'
    '"Great white sharks are some of the only warm-blooded sharks."\n'
)


def build_csqe_prompt(query, retrieved_docs_truncated):
    """
    CSQE corpus-grounded prompt (paper Table 2).
    LLM should EXTRACT sentences from retrieved docs, not freely generate.
    """
    docs_str = '\n'.join(
        f'{i+1}. {doc}' for i, doc in enumerate(retrieved_docs_truncated)
    )
    prompt = (
        f'{CSQE_ONE_SHOT}\n'
        f'Query: "{query}"\n'
        f'Retrieved documents:\n{docs_str}\n'
        f'You will begin by examining the initially retrieved documents and identifying '
        f'the ones that are relevant, even partially, to the query. Once the relevant '
        f'documents are identified, you will extract the key sentences from each document '
        f'that contribute to their relevance. Respond in Arabic only.'
    )
    return prompt


def build_blind_prompt(query):
    """
    Blind (KEQE/Query2Doc) prompt â€” paper Table 1.
    System prompt handles the task; user message is just the query.
    """
    return query


print('Prompt templates defined')
print(f'BLIND_SYSTEM: {BLIND_SYSTEM[:60]}...')
print(f'CSQE_SYSTEM:  {CSQE_SYSTEM[:60]}...')


In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ── Clean up any leftover VRAM ──
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('=== GPU before model load ===')
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {torch.cuda.get_device_name(0)} ({total:.0f}GB)')

print(f'\nLoading Aya Expanse 8B in BF16 (no quantization)...')
print(f'Model: {CONFIG_BASE['model_name']}')

# BF16: ~16GB VRAM. On A100 80GB this leaves ~64GB for KV cache + batching.
# Faster than 4-bit NF4 because no dequantization overhead.
# A100 has native BF16 tensor cores.
tokenizer = AutoTokenizer.from_pretrained(CONFIG_BASE['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # Required for decoder-only batch generation

model = AutoModelForCausalLM.from_pretrained(
    CONFIG_BASE['model_name'],
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)
model.eval()

print(f'\nAya Expanse 8B loaded (BF16)')
print(f'Model device: {model.device}')
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {allocated:.1f}GB / {total:.1f}GB used')
    print(f'Free for batching: {total - allocated:.1f}GB')

# Reload first-pass if needed (e.g., after runtime restart)
if 'firstpass_results' not in dir():
    fp_path = '/content/drive/MyDrive/exp_013_firstpass.pkl'
    print(f'\nReloading first-pass results from {fp_path}')
    with open(fp_path, 'rb') as f:
        firstpass_results = pickle.load(f)
    print(f'Loaded {len(firstpass_results)} queries')

In [ ]:
class CSQEEnhancer:
    """
    Corpus-Steered Query Expansion (CSQE) enhancer.
    Supports both sequential (for sanity checks) and batched (for full runs) generation.
    """

    def __init__(self, model, tokenizer, firstpass_results, config):
        self.model = model
        self.tokenizer = tokenizer
        self.firstpass = firstpass_results
        self.config = config

    def get_retrieved_docs(self, qid):
        """Get pre-computed BM25 first-pass docs, truncated."""
        raw_docs = self.firstpass.get(qid, [])
        docs = []
        for d in raw_docs:
            truncated = truncate_to_tokens(
                d['text'],
                max_tokens=self.config['doc_truncation_tokens'],
                tokenizer=self.tokenizer
            )
            docs.append({
                'docid': d['docid'],
                'score': d['score'],
                'text': truncated,
                'meta': parse_docid(d['docid'])
            })
        return docs

    # ── Sequential generation (for sanity checks) ──

    def generate_samples(self, system_prompt, user_prompt, n_samples, temperature=None):
        """
        Generate n_samples in ONE forward pass via num_return_sequences.
        Used for sanity checks (single query at a time).
        """
        if temperature is None:
            temperature = self.config['temperature']

        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ]

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = self.tokenizer(text, return_tensors='pt').to(self.model.device)
        input_len = inputs.input_ids.shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.config['max_new_tokens'],
                temperature=temperature,
                top_p=self.config['top_p'],
                do_sample=True,
                num_return_sequences=n_samples,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        samples = []
        for seq in outputs:
            generated = seq[input_len:]
            decoded = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
            samples.append(decoded)
        return samples

    def enhance(self, qid, query):
        """Sequential CSQE for a single query (sanity checks)."""
        retrieved_docs = self.get_retrieved_docs(qid)
        doc_texts = [d['text'] for d in retrieved_docs if d['text']]

        # Guard: num_return_sequences=0 is invalid in HuggingFace
        if self.config['num_corpus_samples'] > 0:
            corpus_user_prompt = build_csqe_prompt(query, doc_texts)
            corpus_expansions = self.generate_samples(
                CSQE_SYSTEM, corpus_user_prompt,
                n_samples=self.config['num_corpus_samples'], temperature=1.0
            )
        else:
            corpus_expansions = []

        if self.config['num_blind_samples'] > 0:
            blind_user_prompt = build_blind_prompt(query)
            blind_expansions = self.generate_samples(
                BLIND_SYSTEM, blind_user_prompt,
                n_samples=self.config['num_blind_samples'], temperature=1.0
            )
        else:
            blind_expansions = []

        alpha = self.config['query_repetition']
        all_expansions = corpus_expansions + blind_expansions
        final_query = (query + ' ') * alpha + ' '.join(all_expansions)

        return {
            'original': query,
            'retrieved_docids': [d['docid'] for d in retrieved_docs],
            'retrieved_doc_texts': [d['text'] for d in retrieved_docs],
            'corpus_expansions': corpus_expansions,
            'blind_expansions': blind_expansions,
            'enhanced': final_query,
        }

    # ── Batched generation (for full runs) ──

    def batch_generate(self, system_prompt, user_prompts, batch_size,
                       temperature=None, max_length=2048):
        """
        Generate 1 sample per prompt for a list of prompts, in mini-batches.
        Same pattern as our proven Aya Query2Doc notebook (enhance_batch_parallel).
        Returns list of decoded strings, one per prompt.
        """
        if temperature is None:
            temperature = self.config['temperature']

        # Step 1: build chat-formatted texts (not batchable — loop)
        texts = []
        for user_prompt in user_prompts:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            texts.append(text)

        # Step 2: process in mini-batches
        all_outputs = []
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]

            inputs = self.tokenizer(
                batch_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(self.model.device)

            input_length = inputs.input_ids.shape[1]

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.config['max_new_tokens'],
                    temperature=temperature,
                    top_p=self.config['top_p'],
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            decoded = self.tokenizer.batch_decode(
                outputs[:, input_length:],
                skip_special_tokens=True
            )
            all_outputs.extend([d.strip() for d in decoded])

        return all_outputs

    def batch_enhance(self, qids, queries):
        """
        Full CSQE pipeline for a batch of queries.
        Runs num_corpus_samples + num_blind_samples batched generate passes.
        Returns list of result dicts.
        """
        n = len(qids)
        corpus_bs = self.config['corpus_batch_size']
        blind_bs = self.config['blind_batch_size']
        max_corpus_len = self.config['max_corpus_prompt_length']
        max_blind_len = self.config['max_blind_prompt_length']
        n_corpus = self.config['num_corpus_samples']
        n_blind = self.config['num_blind_samples']

        # ── Step 1: Build all prompts ──
        corpus_prompts = []
        blind_prompts = []
        all_retrieved = []
        for qid, query in zip(qids, queries):
            docs = self.get_retrieved_docs(qid)
            all_retrieved.append(docs)
            doc_texts = [d['text'] for d in docs if d['text']]
            corpus_prompts.append(build_csqe_prompt(query, doc_texts))
            blind_prompts.append(build_blind_prompt(query))

        # ── Step 2: Corpus expansions (n_corpus passes, one sample per pass) ──
        corpus_all_samples = [[] for _ in range(n)]
        for _ in range(n_corpus):
            samples = self.batch_generate(
                CSQE_SYSTEM, corpus_prompts, corpus_bs,
                temperature=1.0, max_length=max_corpus_len
            )
            for i, s in enumerate(samples):
                corpus_all_samples[i].append(s)

        # ── Step 3: Blind expansions (n_blind passes, one sample per pass) ──
        blind_all_samples = [[] for _ in range(n)]
        for _ in range(n_blind):
            samples = self.batch_generate(
                BLIND_SYSTEM, blind_prompts, blind_bs,
                temperature=1.0, max_length=max_blind_len
            )
            for i, s in enumerate(samples):
                blind_all_samples[i].append(s)

        # ── Step 4: Assemble results ──
        alpha = self.config['query_repetition']
        results = []
        for i in range(n):
            corpus_exps = corpus_all_samples[i]
            blind_exps = blind_all_samples[i]
            all_exps = corpus_exps + blind_exps
            final = (queries[i] + ' ') * alpha + ' '.join(all_exps)

            results.append({
                'qid': qids[i],
                'original': queries[i],
                'retrieved_docids': [d['docid'] for d in all_retrieved[i]],
                'retrieved_doc_texts': [d['text'] for d in all_retrieved[i]],
                'corpus_expansions': corpus_exps,
                'blind_expansions': blind_exps,
                'enhanced': final,
            })

        return results


# CONFIG_BASE used here — CONFIG_C and CONFIG_D are created below per ablation
enhancer = CSQEEnhancer(
    model=model, tokenizer=tokenizer,
    firstpass_results=firstpass_results, config=CONFIG_BASE
)

print('CSQEEnhancer ready')
print(f'  Sequential: enhance(qid, query) — for sanity checks')
print(f'  Batched: batch_enhance(qids, queries) — for full run')
print(f'  Corpus batch size: {CONFIG_BASE["corpus_batch_size"]}')
print(f'  Blind batch size: {CONFIG_BASE["blind_batch_size"]}')

---
## Ablation 013c — Corpus-Only (N1=4, N2=0)

All 4 expansions come from corpus-grounded extraction (LLM reads retrieved docs).
No blind/parametric expansions. Tests whether corpus grounding alone explains the gain.

In [ ]:
CONFIG_C = {
    **CONFIG_BASE,
    'num_corpus_samples': 4,
    'num_blind_samples': 0,
    'exp_id': 'exp_013c',
    'output_pkl': 'results/enhanced_queries/exp_013c_corpus_only.pkl',
    'output_trec_bm25': 'results/exp_013c_corpus_only_bm25.txt',
    'run_name': 'exp_013c_corpus_only',
    'checkpoint_path': '/content/drive/MyDrive/exp_013c_checkpoint.pkl',
}

print('CONFIG_C (corpus-only):')
print(f'  num_corpus_samples: {CONFIG_C["num_corpus_samples"]}')
print(f'  num_blind_samples:  {CONFIG_C["num_blind_samples"]}')
print(f'  query_repetition:   {CONFIG_C["query_repetition"]}')
print(f'  output_pkl:         {CONFIG_C["output_pkl"]}')

In [ ]:
# ── 013c Sanity check (3 queries) ──────────────────────────────
enhancer_c = CSQEEnhancer(
    model=model, tokenizer=tokenizer,
    firstpass_results=firstpass_results, config=CONFIG_C
)

print('013c sanity check (3 queries)...')
for qid in query_ids[:3]:
    q = topics[qid]['title']
    r = enhancer_c.enhance(qid, q)
    print(f'QID {qid}: {q}')
    print(f'  corpus_exp 1 ({len(r["corpus_expansions"][0])}ch): {r["corpus_expansions"][0][:100]}')
    print(f'  corpus_exp 2 ({len(r["corpus_expansions"][1])}ch): {r["corpus_expansions"][1][:100]}')
    print(f'  blind_exps: {r["blind_expansions"]}')
    print()

In [ ]:
# ── 013c Full generation ────────────────────────────────────────
import os

start_idx_c = 0
results_c = []

if os.path.exists(CONFIG_C['checkpoint_path']):
    print(f'Checkpoint found, resuming...')
    with open(CONFIG_C['checkpoint_path'], 'rb') as f:
        ckpt = pickle.load(f)
    results_c = ckpt['results']
    start_idx_c = len(results_c)
    print(f'Resuming from {start_idx_c}/{len(query_ids)}')
else:
    print('Starting 013c from scratch')

remaining_c = query_ids[start_idx_c:]
remaining_c_texts = query_texts[start_idx_c:]
macro_bs = CONFIG_C['macro_batch_size']
start_time = time.time()

for batch_start in range(0, len(remaining_c), macro_bs):
    batch_end = min(batch_start + macro_bs, len(remaining_c))
    batch_qids = remaining_c[batch_start:batch_end]
    batch_qs = remaining_c_texts[batch_start:batch_end]
    batch_num = batch_start // macro_bs + 1
    total_batches = (len(remaining_c) + macro_bs - 1) // macro_bs
    print(f'Batch {batch_num}/{total_batches} ({len(batch_qids)} queries)...')
    try:
        batch_results = enhancer_c.batch_enhance(batch_qids, batch_qs)
        results_c.extend(batch_results)
    except Exception as e:
        print(f'  Batch error: {e}, falling back to sequential')
        for qid, query in zip(batch_qids, batch_qs):
            try:
                r = enhancer_c.enhance(qid, query)
                r['qid'] = qid
                results_c.append(r)
            except Exception as e2:
                alpha = CONFIG_C['query_repetition']
                results_c.append({'qid': qid, 'original': query,
                    'retrieved_docids': [], 'retrieved_doc_texts': [],
                    'corpus_expansions': [''] * CONFIG_C['num_corpus_samples'],
                    'blind_expansions': [], 'enhanced': (query + ' ') * alpha, 'error': str(e2)})
    with open(CONFIG_C['checkpoint_path'], 'wb') as f:
        pickle.dump({'results': results_c, 'config': CONFIG_C}, f)
    elapsed = time.time() - start_time
    done = len(results_c) - start_idx_c
    rate = done / (elapsed / 60) if elapsed > 0 else 0
    remaining_q = len(query_ids) - len(results_c)
    print(f'  {len(results_c)}/{len(query_ids)} | {rate:.0f} q/min | ~{remaining_q / rate:.0f}min left' if rate > 0 else f'  {len(results_c)}/{len(query_ids)}')

print(f'013c done: {len(results_c)} queries in {(time.time()-start_time)/60:.1f} min')

In [ ]:
# ── 013c Save pkl ───────────────────────────────────────────────
import os
os.makedirs('results/enhanced_queries', exist_ok=True)

output_c = {
    'query_ids': [r['qid'] for r in results_c],
    'original':  [r['original'] for r in results_c],
    'enhanced':  [r['enhanced'] for r in results_c],
    'model': CONFIG_C['model_name'],
    'config': CONFIG_C,
    'full_results': results_c,
    'stats': {
        'total_queries': len(results_c),
        'avg_enhanced_len': sum(len(r['enhanced']) for r in results_c) / len(results_c),
        'error_count': sum(1 for r in results_c if 'error' in r),
    }
}

with open(CONFIG_C['output_pkl'], 'wb') as f:
    pickle.dump(output_c, f)

# Copy to Drive
drive_path_c = f"/content/drive/MyDrive/{CONFIG_C['exp_id']}_csqe_aya_8b.pkl"
import shutil
shutil.copy(CONFIG_C['output_pkl'], drive_path_c)
print(f'013c saved: {CONFIG_C["output_pkl"]}')
print(f'013c Drive: {drive_path_c}')
print(f'Errors: {output_c["stats"]["error_count"]}')

---
## Ablation 013d — Blind-Only (N1=0, N2=4)

All 4 expansions are blind/parametric (no corpus docs shown to LLM).
This is equivalent to Query2Doc with 4 samples instead of 1.
Tests whether the CSQE corpus grounding adds anything beyond just having more samples.

In [ ]:
CONFIG_D = {
    **CONFIG_BASE,
    'num_corpus_samples': 0,
    'num_blind_samples': 4,
    'exp_id': 'exp_013d',
    'output_pkl': 'results/enhanced_queries/exp_013d_blind_only.pkl',
    'output_trec_bm25': 'results/exp_013d_blind_only_bm25.txt',
    'run_name': 'exp_013d_blind_only',
    'checkpoint_path': '/content/drive/MyDrive/exp_013d_checkpoint.pkl',
}

print('CONFIG_D (blind-only):')
print(f'  num_corpus_samples: {CONFIG_D["num_corpus_samples"]}')
print(f'  num_blind_samples:  {CONFIG_D["num_blind_samples"]}')
print(f'  query_repetition:   {CONFIG_D["query_repetition"]}')
print(f'  output_pkl:         {CONFIG_D["output_pkl"]}')

In [ ]:
# ── 013d Sanity check (3 queries) ──────────────────────────────
enhancer_d = CSQEEnhancer(
    model=model, tokenizer=tokenizer,
    firstpass_results=firstpass_results, config=CONFIG_D
)

print('013d sanity check (3 queries)...')
for qid in query_ids[:3]:
    q = topics[qid]['title']
    r = enhancer_d.enhance(qid, q)
    print(f'QID {qid}: {q}')
    print(f'  corpus_exps: {r["corpus_expansions"]}')
    print(f'  blind_exp 1 ({len(r["blind_expansions"][0])}ch): {r["blind_expansions"][0][:100]}')
    print(f'  blind_exp 2 ({len(r["blind_expansions"][1])}ch): {r["blind_expansions"][1][:100]}')
    print()

In [ ]:
# ── 013d Full generation ────────────────────────────────────────
start_idx_d = 0
results_d = []

if os.path.exists(CONFIG_D['checkpoint_path']):
    print(f'Checkpoint found, resuming...')
    with open(CONFIG_D['checkpoint_path'], 'rb') as f:
        ckpt = pickle.load(f)
    results_d = ckpt['results']
    start_idx_d = len(results_d)
    print(f'Resuming from {start_idx_d}/{len(query_ids)}')
else:
    print('Starting 013d from scratch')

remaining_d = query_ids[start_idx_d:]
remaining_d_texts = query_texts[start_idx_d:]
start_time = time.time()

for batch_start in range(0, len(remaining_d), macro_bs):
    batch_end = min(batch_start + macro_bs, len(remaining_d))
    batch_qids = remaining_d[batch_start:batch_end]
    batch_qs = remaining_d_texts[batch_start:batch_end]
    batch_num = batch_start // macro_bs + 1
    total_batches = (len(remaining_d) + macro_bs - 1) // macro_bs
    print(f'Batch {batch_num}/{total_batches} ({len(batch_qids)} queries)...')
    try:
        batch_results = enhancer_d.batch_enhance(batch_qids, batch_qs)
        results_d.extend(batch_results)
    except Exception as e:
        print(f'  Batch error: {e}, falling back to sequential')
        for qid, query in zip(batch_qids, batch_qs):
            try:
                r = enhancer_d.enhance(qid, query)
                r['qid'] = qid
                results_d.append(r)
            except Exception as e2:
                alpha = CONFIG_D['query_repetition']
                results_d.append({'qid': qid, 'original': query,
                    'retrieved_docids': [], 'retrieved_doc_texts': [],
                    'corpus_expansions': [], 'blind_expansions': [''] * CONFIG_D['num_blind_samples'],
                    'enhanced': (query + ' ') * alpha, 'error': str(e2)})
    with open(CONFIG_D['checkpoint_path'], 'wb') as f:
        pickle.dump({'results': results_d, 'config': CONFIG_D}, f)
    elapsed = time.time() - start_time
    done = len(results_d) - start_idx_d
    rate = done / (elapsed / 60) if elapsed > 0 else 0
    remaining_q = len(query_ids) - len(results_d)
    print(f'  {len(results_d)}/{len(query_ids)} | {rate:.0f} q/min | ~{remaining_q / rate:.0f}min left' if rate > 0 else f'  {len(results_d)}/{len(query_ids)}')

print(f'013d done: {len(results_d)} queries in {(time.time()-start_time)/60:.1f} min')

In [ ]:
# ── 013d Save pkl ───────────────────────────────────────────────
output_d = {
    'query_ids': [r['qid'] for r in results_d],
    'original':  [r['original'] for r in results_d],
    'enhanced':  [r['enhanced'] for r in results_d],
    'model': CONFIG_D['model_name'],
    'config': CONFIG_D,
    'full_results': results_d,
    'stats': {
        'total_queries': len(results_d),
        'avg_enhanced_len': sum(len(r['enhanced']) for r in results_d) / len(results_d),
        'error_count': sum(1 for r in results_d if 'error' in r),
    }
}

with open(CONFIG_D['output_pkl'], 'wb') as f:
    pickle.dump(output_d, f)

drive_path_d = f"/content/drive/MyDrive/{CONFIG_D['exp_id']}_csqe_aya_8b.pkl"
shutil.copy(CONFIG_D['output_pkl'], drive_path_d)
print(f'013d saved: {CONFIG_D["output_pkl"]}')
print(f'013d Drive: {drive_path_d}')
print(f'Errors: {output_d["stats"]["error_count"]}')

print('\n=== DONE ===')
print('Upload both pkls to evaluate_enhanced_queries.ipynb or phase4_quick_wins.ipynb Section 11')
print(f'013c: {CONFIG_C["output_pkl"]}')
print(f'013d: {CONFIG_D["output_pkl"]}')